In [1]:
import tensorflow as tf
import numpy as np
import tensorflow_recommenders as tfrs

In [2]:
def generate_embedding(item, item_model):
  
      item_id = item['STOCKCODE'] 
      item_name = item['STOCKNAME']
      item_gics = item['GICS']

      embedding = item_model([item_id, item_name, item_gics])
      return embedding


In [3]:
class ItemModel(tf.keras.Model):
    def __init__(
        self,
        unique_item_ids,
        unique_item_names,
        unique_item_gics,
        # map_ = False

        ):
        super().__init__()

        self.max_tokens = 10000
        self.unique_item_ids = unique_item_ids
        self.unique_item_names = unique_item_names
        self.unique_item_gics = unique_item_gics
        # self.map_ = map_

        

        self.embed_item_id = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = self.unique_item_ids,
                mask_token =None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(self.unique_item_ids)+1,
                output_dim = 8 #32
            )
        ])

        self.embed_item_gics = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = unique_item_gics,
                mask_token = None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(unique_item_gics)+1,
                output_dim = 8 #len(unique_item_gics)
            )
        ])


        self.textvectorizer = tf.keras.layers.TextVectorization(
            max_tokens = self.max_tokens
        )

        self.embed_item_name = tf.keras.Sequential([
            self.textvectorizer,

            tf.keras.layers.Embedding(
                input_dim = self.max_tokens,
                output_dim = 16,
                mask_zero = True
            ),

            tf.keras.layers.GlobalAveragePooling1D() # reduces dimensionality to 1d (embedding layer embeddeds each word in a title one by one)
        ])

        self.textvectorizer.adapt(self.unique_item_names)
    
    def call(self, inputs):

        item_id, item_name, item_gics = inputs['STOCKCODE'], inputs['STOCKNAME'], inputs['GICS']

        return tf.concat([
            self.embed_item_id(item_id),
            self.embed_item_name(item_name),
            self.embed_item_gics(item_gics)
        ],
        axis = 1)

In [4]:
class UserModel(tf.keras.Model):
    def __init__(
        self,
        unique_item_ids):

        super().__init__()

        self.unique_item_ids = unique_item_ids
        
        self.embed_user_id = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = self.unique_item_ids,
                mask_token = None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(self.unique_item_ids)+1,
                output_dim = 32
            ),
            tf.keras.layers.GlobalAveragePooling1D()
    
        ])
        
    def call(self, inputs):

        user_id = inputs["USER_ID"]

        return self.embed_user_id(user_id)


In [5]:
class Retriever(tfrs.models.Model):

  def __init__(
    self,
    # use_timestamp,
    portfolios
    ):

    super().__init__()

    # self.use_timestamp = use_timestamp
    self.portfolios = portfolios

    self.items_ids = self.portfolios.batch(10000).map(lambda x: x["STOCKCODE"])
    self.item_names = self.portfolios.batch(10000).map(lambda x: x["STOCKNAME"])
    self.item_GICS = self.portfolios.batch(10000).map(lambda x: x["GICS"])

    self.unique_item_ids = np.unique(np.concatenate(list(self.items_ids)))
    self.unique_item_names = np.unique(np.concatenate(list(self.item_names)))
    self.unique_item_gics = np.unique(np.concatenate(list(self.item_GICS)))

    # need these to initialize timestamp embedding layers in future steps

    self.timestamps = np.concatenate(list(self.portfolios.map(lambda x: x["UNIX_TS"]).batch(100)))

    self.max_timestamp = self.timestamps.max()
    self.min_timestamp = self.timestamps.min()

    self.timestamp_buckets = np.linspace(
        self.min_timestamp, self.max_timestamp, num=1000,
    )

    self.item_model = ItemModel(
      unique_item_ids = self.unique_item_ids,
      unique_item_names = self.unique_item_names,
      unique_item_gics = self.unique_item_gics
    )

    self.user_model = UserModel(
      unique_item_ids = self.unique_item_ids
    )

    self.retrieval_metrics = tfrs.metrics.FactorizedTopK(
      candidates= self.portfolios.batch(128).map(lambda x:self.item_model(x)),
      ks = [10]
    )

    self.task = tfrs.tasks.Retrieval(
      metrics = self.retrieval_metrics
    )

  def compute_loss(self, features, training=False) -> tf.Tensor:
    user_embeddings = self.user_model(features)

    item_embeddings = self.item_model(features)
    
    return self.task(
      query_embeddings = user_embeddings,
      candidate_embeddings = item_embeddings
      )

# Training

In [6]:
import os

import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from tqdm.keras import TqdmCallback
from datetime import datetime


c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
base_loc = r'D:\dev work\recommender systems\ATRAD_CARS'

train_ds = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\retriver_train").cache() #data\ratings_train
test_ds = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\retriver_test").cache()
portfolios = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\portfolios").cache()


In [8]:
next(iter(train_ds))

{'STOCKNAME': <tf.Tensor: shape=(), dtype=string, numpy=b'VALLIBEL ONE PLC'>,
 'GICS': <tf.Tensor: shape=(), dtype=string, numpy=b'Utilities'>,
 'USER_ID': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL', b'RCL',
        b'CALT', b'PLC'], dtype=object)>,
 'STOCKCODE': <tf.Tensor: shape=(), dtype=string, numpy=b'VONE'>,
 'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'BMS-10544-LC/00'>,
 'RATING': <tf.Tensor: shape=(), dtype=float32, numpy=3.0>,
 'UNIX_TS': <tf.Tensor: shape=(), dtype=float32, numpy=1641148200.0>}

In [9]:
model = Retriever(
    portfolios = portfolios
    )

In [10]:
log_dir = os.path.join(base_loc ,"logs/fit/retriever_port_v2_fixed_max_port_size_50_useridseq/" + "retriever_v3_" + datetime.now().strftime("%Y%m%d_%H%M%S"))

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    embeddings_freq = 1,
    write_images = True)

model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.3))

train_ds = train_ds.shuffle(100000).batch(512) #.cache()
test_ds = test_ds.batch(256)

In [11]:
ex_train_ds = next(iter(train_ds))
ex_train_ds

{'STOCKNAME': <tf.Tensor: shape=(512,), dtype=string, numpy=
 array([b'BROWNS INVESTMENTS PLC', b'CHEVRON LUBRICANTS  LANKA  PLC',
        b'SAMPATH BANK PLC', b'BROWN & COMPANY PLC', b'LOLC FINANCE PLC',
        b'ASIRI HOSPITAL HOLDINGS PLC', b'JOHN KEELLS HOLDINGS PLC',
        b'CAPITAL ALLIANCE PLC', b'FIRST CAPITAL HOLDINGS PLC',
        b'EDEN HOTEL LANKA PLC', b'ACL CABLES PLC',
        b'CEYLON TOBACCO COMPANY PLC', b'RENUKA AGRI FOODS PLC',
        b'AGALAWATTE  PLANTATIONS  PLC', b'SAMPATH BANK PLC',
        b'CHEVRON LUBRICANTS  LANKA  PLC', b'CENTRAL INDUSTRIES PLC',
        b'SUNSHINE HOLDINGS PLC', b'LANKA IOC PLC',
        b'CEYLON GRAIN ELEVATORS PLC', b'ACL CABLES PLC',
        b'PAN ASIA BANKING CORPORATION PLC',
        b'PRIME LANDS RESIDENCIES PLC', b'ALLIANCE FINANCE COMPANY PLC',
        b'AMBEON CAPITAL PLC', b'NATIONAL DEVELOPMENT BANK PLC',
        b'UNISYST ENGINEERING PLC', b'KOTAGALA PLANTATIONS PLC',
        b'LANKA TILES PLC', b'VALLIBEL FINANCE PLC',
  

In [12]:
user_model_t = model.user_model
user_model_t(ex_train_ds)

<tf.Tensor: shape=(512, 32), dtype=float32, numpy=
array([[ 0.00340358, -0.00363942,  0.00026852, ..., -0.0003428 ,
         0.00653207, -0.00663579],
       [-0.00131119,  0.00694213, -0.00336843, ..., -0.01021491,
         0.00559874, -0.00966456],
       [-0.0194988 , -0.00483142, -0.00365238, ..., -0.01797352,
        -0.00967342,  0.00216247],
       ...,
       [-0.00165761,  0.00079621, -0.00015296, ..., -0.00085503,
         0.00745397,  0.00362373],
       [-0.00044152, -0.0084608 ,  0.00297218, ..., -0.00386917,
         0.00485621, -0.0062717 ],
       [ 0.00555183,  0.0021129 , -0.01001027, ..., -0.01459386,
         0.00095213,  0.00443232]], dtype=float32)>

In [21]:
# portfolios

unique_item_ids = np.unique(list((portfolios.map(lambda x: x['STOCKCODE'])).as_numpy_iterator()))
# len(np.unique(list((portfolios.map(lambda x: x['STOCKCODE'])).as_numpy_iterator())))

In [28]:
strlookuplayer = tf.keras.layers.StringLookup(
                vocabulary = unique_item_ids,
                mask_token = None
                )
                
test_user_id = next(iter(portfolios.batch(1).map(lambda x: x['USER_ID'])))
strlookuplayer(test_user_id)

<tf.Tensor: shape=(1, 10), dtype=int64, numpy=array([[ 69,  99, 221, 185, 100, 218, 241, 208,  48, 203]], dtype=int64)>

In [29]:
test_user_id = next(iter(portfolios.batch(1).map(lambda x: x['USER_ID'])))
str_lookup_output = strlookuplayer(test_user_id)

embedding_layer = tf.keras.layers.Embedding(
                input_dim = len(unique_item_ids)+1,
                output_dim = 32
                )

embedding_layer(str_lookup_output)

<tf.Tensor: shape=(1, 10, 32), dtype=float32, numpy=
array([[[-0.04398668, -0.01548245,  0.0408854 ,  0.02802174,
          0.02810079, -0.00217779, -0.04282544, -0.01154177,
          0.04802766, -0.02303066,  0.00464852,  0.04267236,
          0.01984071,  0.00949011,  0.03687047, -0.0430644 ,
          0.0369198 ,  0.01326932,  0.04961321,  0.04558   ,
          0.03461678,  0.02895189,  0.04688429,  0.02952055,
          0.04800857,  0.02640421, -0.00604888,  0.04453938,
          0.03045586, -0.00478582, -0.02220362,  0.03298363],
        [ 0.01764897, -0.042999  , -0.0140953 ,  0.03794796,
         -0.02046531,  0.014469  ,  0.03311909,  0.02739373,
          0.01172789, -0.03979687, -0.00273724,  0.0384095 ,
          0.00743375, -0.04608826, -0.04903994,  0.04608593,
          0.01181828, -0.02231455,  0.00706031,  0.02132506,
         -0.00326761, -0.00330027,  0.03212724,  0.04242977,
         -0.04846964,  0.03485279,  0.01021342, -0.01995561,
          0.01199347,  0.014236

In [31]:
test_user_id = next(iter(portfolios.batch(1).map(lambda x: x['USER_ID'])))
str_lookup_output = strlookuplayer(test_user_id)
embedding_layer_output = embedding_layer(str_lookup_output)

pooling_layer = tf.keras.layers.GlobalAveragePooling1D()
pooling_layer(embedding_layer_output)


<tf.Tensor: shape=(1, 32), dtype=float32, numpy=
array([[ 0.00887265, -0.01186202, -0.00128829,  0.01328382,  0.01589919,
         0.00457747, -0.00261044,  0.00158563,  0.02331115, -0.01706183,
        -0.01293038,  0.00050502, -0.00232729, -0.00299405, -0.02637031,
        -0.00955961,  0.01999114, -0.00461632,  0.00247488,  0.0194476 ,
         0.00965053,  0.00449963, -0.00926486,  0.02110349,  0.00911644,
         0.00776165,  0.01036273, -0.00122118,  0.00251207,  0.00818959,
        -0.00949168,  0.01712869]], dtype=float32)>

In [ ]:
self.embed_user_id = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = self.unique_item_ids,
                mask_token = None 
            ),
            tf.keras.layers.Embedding(
                input_dim = len(self.unique_item_ids)+1,
                output_dim = 32
            ),
            tf.keras.layers.GlobalAveragePooling1D()
            ])

In [14]:
history = model.fit(
    train_ds, 
    epochs=10, 
    verbose = 0,
    validation_data=test_ds,
    validation_freq=1,
    callbacks=[tensorboard_callback, TqdmCallback(verbose=1)]
    )


KeyboardInterrupt: 

In [ ]:
print(history)

In [ ]:
import sys 
sys.exit()

In [ ]:


#save model
base = r'D:\dev work\recommender systems\ATRAD_CARS\model_weights\{}'.format(datetime.now().strftime("%Y_%m_%d_%M"))

if not os.path.exists(base):
    os.makedirs(base)

model_name = 'retriever_v3_port_v2__fixed_port_size_20_useridseq' 
#model_name = 'retriever_port_v2_hoo'
save_path = os.path.join(base,model_name)

model.save_weights(save_path)

print()
print("saved model @ : {}".format(save_path))



saved model @ : D:\dev work\recommender systems\ATRAD_CARS\model_weights\2024_06_26_51\retriever_v3_port_v2__fixed_port_size_20_useridseq
